# 🏠 House Price Prediction — King County

End-to-end analytics notebook using the King County Housing dataset.  
**Sections:**
1. Data Loading & Inspection
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Preprocessing
4. Model Training & Evaluation (RandomForest + GradientBoosting)
5. Model Persistence

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('Libraries loaded ✓')

---
## Section 1 — Data Loading & Inspection

In [ ]:
# ── Cell 2: Load Data ─────────────────────────────────────────────────────────
DATA_PATH = '../Housing.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# ── Cell 3: Data Types & Memory ──────────────────────────────────────────────
df.info()

In [ ]:
# ── Cell 4: Descriptive Statistics ───────────────────────────────────────────
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# ── Cell 5: Missing Values & Duplicates ──────────────────────────────────────
print('=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nTotal missing: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

---
## Section 2 — Exploratory Data Analysis

In [ ]:
# ── Cell 6: Price Distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price
axes[0].hist(df['price'], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (Raw)', fontsize=13)
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

# Log-transformed price
axes[1].hist(np.log1p(df['price']), bins=80, color='darkorange', edgecolor='white')
axes[1].set_title('Price Distribution (Log-Transformed)', fontsize=13)
axes[1].set_xlabel('log1p(Price)')
axes[1].set_ylabel('Count')

plt.suptitle('Target Variable: House Price', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f'Price skewness (raw): {df["price"].skew():.2f}')
print(f'Price skewness (log): {np.log1p(df["price"]).skew():.2f}')

In [ ]:
# ── Cell 7: Correlation Heatmap ──────────────────────────────────────────────
numeric_cols = df.select_dtypes(include=np.number).drop(columns=['id'], errors='ignore')
corr = numeric_cols.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=15)
plt.tight_layout()
plt.show()

print('\nTop correlations with price:')
print(corr['price'].sort_values(ascending=False).head(10))

In [ ]:
# ── Cell 8: Box Plots — Price vs Categorical Features ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col in zip(axes, ['bedrooms', 'grade', 'condition']):
    temp = df[df[col] <= df[col].quantile(0.99)]  # remove extreme outliers for display
    grouped = [temp[temp[col] == v]['price'].values for v in sorted(temp[col].unique())]
    ax.boxplot(grouped, labels=sorted(temp[col].unique()), patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='darkblue', linewidth=2))
    ax.set_title(f'Price by {col.capitalize()}', fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel('Price (USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

plt.suptitle('Price Distribution by Key Categorical Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 9: Scatter — sqft_living vs price ────────────────────────────────────
fig = px.scatter(
    df, x='sqft_living', y='price', color='grade',
    title='Living Area vs Price (coloured by Grade)',
    labels={'sqft_living': 'Living Area (sqft)', 'price': 'Price (USD)'},
    opacity=0.5, color_continuous_scale='Viridis', height=500
)
fig.update_layout(coloraxis_colorbar_title='Grade')
fig.show()

In [ ]:
# ── Cell 10: Waterfront Premium ──────────────────────────────────────────────
wf = df.groupby('waterfront')['price'].median().reset_index()
wf['waterfront_label'] = wf['waterfront'].map({0: 'No Waterfront', 1: 'Waterfront'})

fig = px.bar(
    wf, x='waterfront_label', y='price', color='waterfront_label',
    title='Median Price: Waterfront vs Non-Waterfront',
    labels={'price': 'Median Price (USD)', 'waterfront_label': ''},
    text_auto='.2s', color_discrete_sequence=['#636EFA', '#EF553B']
)
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# ── Cell 11: Geographic Scatter Map ──────────────────────────────────────────
fig = px.scatter_mapbox(
    df, lat='lat', lon='long', color='price',
    color_continuous_scale='Plasma', size_max=8,
    zoom=9, height=600,
    title='King County — House Prices by Location',
    mapbox_style='open-street-map',
    hover_data={'price': ':,.0f', 'bedrooms': True, 'sqft_living': True}
)
fig.update_layout(coloraxis_colorbar_title='Price (USD)')
fig.show()

In [ ]:
# ── Cell 12: Price vs Year Built ─────────────────────────────────────────────
avg_by_year = df.groupby('yr_built')['price'].median().reset_index()

fig = px.line(
    avg_by_year, x='yr_built', y='price',
    title='Median House Price by Year Built',
    labels={'yr_built': 'Year Built', 'price': 'Median Price (USD)'}
)
fig.update_traces(line_color='steelblue')
fig.show()

---
## Section 3 — Feature Engineering & Preprocessing

**Engineering rationale:**
- `house_age`: captures depreciation effect  
- `renovated_flag`: binary — renovated homes command a premium  
- `basement_flag`: presence of basement as a feature  
- `sale_month`: seasonality in house prices  
- Drop `id` (identifier), parse `date` for month  
- Keep `lat`/`long` instead of `zipcode` (continuous, richer spatial signal)  
- Log-transform `price` to reduce right-skew → models trained on `y_log`

In [ ]:
# ── Cell 13: Feature Engineering ─────────────────────────────────────────────
df_fe = df.copy()

# Parse date
df_fe['date'] = pd.to_datetime(df_fe['date'])
df_fe['sale_month'] = df_fe['date'].dt.month
df_fe['sale_year']  = df_fe['date'].dt.year

# Age features
df_fe['house_age']       = 2015 - df_fe['yr_built']          # dataset is 2014-2015
df_fe['renovated_flag']  = (df_fe['yr_renovated'] > 0).astype(int)
df_fe['basement_flag']   = (df_fe['sqft_basement'] > 0).astype(int)
df_fe['years_since_reno']= df_fe.apply(
    lambda r: 2015 - r['yr_renovated'] if r['yr_renovated'] > 0 else r['house_age'], axis=1
)

# Drop non-informative columns
df_fe.drop(columns=['id', 'date', 'zipcode', 'yr_built', 'yr_renovated'], inplace=True)

print('Feature-engineered shape:', df_fe.shape)
df_fe.head(3)

In [ ]:
# ── Cell 14: Define X and y ───────────────────────────────────────────────────
TARGET = 'price'

X = df_fe.drop(columns=[TARGET])
y = np.log1p(df_fe[TARGET])   # log-transform target

FEATURE_COLUMNS = list(X.columns)
print(f'Features ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}')
print(f'Target (log-transformed): min={y.min():.2f}, max={y.max():.2f}')

In [ ]:
# ── Cell 15: Train / Test Split ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# ── Cell 16: Scale Features ───────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Save scaler and feature columns
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler,          '../models/scaler.pkl')
joblib.dump(FEATURE_COLUMNS, '../models/feature_columns.pkl')
print('Scaler and feature columns saved ✓')

---
## Section 4 — Model Training & Evaluation

In [ ]:
# ── Cell 17: Train RandomForestRegressor ─────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
print('RandomForest trained ✓')

In [ ]:
# ── Cell 18: Train GradientBoostingRegressor ──────────────────────────────────
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
gb.fit(X_train_sc, y_train)
print('GradientBoosting trained ✓')

In [ ]:
# ── Cell 19: Evaluate Both Models ────────────────────────────────────────────
def evaluate_model(name, model, X_test_sc, y_test):
    y_pred_log = model.predict(X_test_sc)
    y_pred     = np.expm1(y_pred_log)   # reverse log
    y_actual   = np.expm1(y_test)
    mae  = mean_absolute_error(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
    r2   = r2_score(y_actual, y_pred)
    return {'Model': name, 'MAE ($)': f'{mae:,.0f}', 'RMSE ($)': f'{rmse:,.0f}', 'R²': f'{r2:.4f}'}

results = [
    evaluate_model('RandomForest',      rf, X_test_sc, y_test),
    evaluate_model('GradientBoosting',  gb, X_test_sc, y_test),
]

results_df = pd.DataFrame(results).set_index('Model')
print('\n=== Model Comparison ===')
display(results_df)

In [ ]:
# ── Cell 20: Pick Best Model ──────────────────────────────────────────────────
rf_r2 = r2_score(np.expm1(y_test), np.expm1(rf.predict(X_test_sc)))
gb_r2 = r2_score(np.expm1(y_test), np.expm1(gb.predict(X_test_sc)))

best_model      = rf if rf_r2 >= gb_r2 else gb
best_model_name = 'RandomForest' if rf_r2 >= gb_r2 else 'GradientBoosting'
print(f'Best model: {best_model_name} (R²={max(rf_r2, gb_r2):.4f})')

joblib.dump(best_model, '../models/model.pkl')
print('Best model saved to ../models/model.pkl ✓')

In [ ]:
# ── Cell 21: Feature Importances ─────────────────────────────────────────────
importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLUMNS)
importances = importances.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 7))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Top-15 Feature Importances — {best_model_name}', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 22: Actual vs Predicted ─────────────────────────────────────────────
y_pred_best = np.expm1(best_model.predict(X_test_sc))
y_actual    = np.expm1(y_test)

fig = px.scatter(
    x=y_actual, y=y_pred_best,
    labels={'x': 'Actual Price (USD)', 'y': 'Predicted Price (USD)'},
    title=f'Actual vs Predicted — {best_model_name}',
    opacity=0.4
)
# Perfect prediction reference line
line_range = [y_actual.min(), y_actual.max()]
fig.add_trace(go.Scatter(x=line_range, y=line_range, mode='lines',
                         line=dict(color='red', dash='dash'), name='Perfect Fit'))
fig.show()

In [ ]:
# ── Cell 23: Residuals Distribution ──────────────────────────────────────────
residuals = y_actual - y_pred_best

fig = px.histogram(
    x=residuals, nbins=80,
    title='Residuals Distribution (Actual − Predicted)',
    labels={'x': 'Residual (USD)'},
    color_discrete_sequence=['darkorange']
)
fig.add_vline(x=0, line_dash='dash', line_color='red')
fig.show()

print(f'Mean residual: ${residuals.mean():,.0f}')
print(f'Std  residual: ${residuals.std():,.0f}')

In [ ]:
# ── Cell 24: Save RMSE for Streamlit confidence interval ─────────────────────
rmse_val = np.sqrt(mean_squared_error(y_actual, y_pred_best))
joblib.dump(rmse_val, '../models/rmse.pkl')
print(f'Test RMSE: ${rmse_val:,.0f}')
print('All artefacts saved to ../models/ ✓')
print('\n=== DONE ===')